# 08 — Comparing random and spatial model evaluation

Prediction models can look stronger when training and test locations are randomly mixed across the same city. Nearby observations may share urban context, and every borough can appear in both sets. This notebook measures that effect by comparing random five-part evaluation with the borough-based evaluation used in the main analysis.

The comparison uses 12 representative models spanning the PTAL baseline, individual and combined representations, the compact EPC controls and the richer EPC reference. Both evaluation approaches use the same Ridge model, preprocessing and penalty range. Their test partitions answer different questions: random division describes interpolation among intermingled London locations, whereas borough division describes transfer to held-out administrative areas.

## Main findings

All 12 models obtain higher scores under random division than under borough hold-out. For PTAL, random evaluation increases mean R² by approximately 0.0245 to 0.0334 depending on the model. For EPC, the increase is approximately 0.0086 to 0.0219.

The order of the models remains unchanged. Random evaluation therefore makes absolute performance appear more optimistic but does not overturn the main comparison between representations. The borough-based results remain the principal dissertation estimates, while the random results demonstrate how validation design changes the apparent level of predictive performance.

In [ ]:
# Connect Google Drive and load packages for the validation-design comparison.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import gc
import time
import hashlib
import platform
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FINAL_CODE_DIR = Path("/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE")
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({
    name: getattr(_config, name)
    for name in dir(_config)
    if not name.startswith("_")
})

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

print("Python:", platform.python_version())
print("numpy:", np.__version__, "pandas:", pd.__version__, "sklearn:", sklearn.__version__)
print("Canonical table:", FINAL_MODEL_TABLE_PATH, FINAL_MODEL_TABLE_PATH.exists())
print("Notebook-07 results:", INCREMENTAL_RESULTS_PATH, INCREMENTAL_RESULTS_PATH.exists())
print("Notebook-07 predictions:", INCREMENTAL_PREDICTIONS_PATH, INCREMENTAL_PREDICTIONS_PATH.exists())
print("Notebook-07 run spec:", INCREMENTAL_RUN_SPEC_PATH, INCREMENTAL_RUN_SPEC_PATH.exists())
print("Notebook-07 audit:", INCREMENTAL_AUDIT_PATH, INCREMENTAL_AUDIT_PATH.exists())

for required_path in [
    FINAL_MODEL_TABLE_PATH,
    FEATURE_MANIFEST_JSON_PATH,
    INCREMENTAL_RESULTS_PATH,
    INCREMENTAL_PREDICTIONS_PATH,
    INCREMENTAL_RUN_SPEC_PATH,
    INCREMENTAL_AUDIT_PATH,
]:
    assert required_path.exists(), f"Missing prerequisite: {required_path}"

## 1. Load the established models and spatial results

The 12 models are reconstructed from the same feature manifest and model definitions used in the controls analysis. The saved borough results provide the spatial comparator, avoiding unnecessary refitting.

In [ ]:
# Read the common models and the existing borough-based results.
df = pd.read_parquet(FINAL_MODEL_TABLE_PATH)
with open(FEATURE_MANIFEST_JSON_PATH, "r") as f:
    manifest = json.load(f)
with open(INCREMENTAL_RUN_SPEC_PATH, "r") as f:
    source_07_spec = json.load(f)
with open(INCREMENTAL_AUDIT_PATH, "r") as f:
    source_07_audit = json.load(f)

assert source_07_audit["integrity_gate_pass"] is True
assert source_07_audit["interpretation_gate_pass"] is True

target_col = manifest["target_column"]
group_col = manifest["group_column"]
categorical_master = set(manifest["categorical_columns"])
feature_sets = manifest["feature_sets"]

assert len(df) == 26597
assert df["sample_id"].is_unique
assert df[target_col].notna().all()
assert df[group_col].notna().all()
assert df[group_col].nunique() == 33
assert set(df["task"].unique()) == {"PTAL", "EPC"}

df = df.sort_values(["task", "sample_id"], kind="mergesort").reset_index(drop=True)
task_counts = df["task"].value_counts().to_dict()
assert task_counts == {"EPC": 20000, "PTAL": 6597}

model_key_frame = df[["sample_id", "task", group_col, target_col]].copy()
model_key_hash = hashlib.sha256(
    pd.util.hash_pandas_object(model_key_frame, index=False).values.tobytes()
).hexdigest()
manifest_hash = hashlib.sha256(
    json.dumps(manifest, sort_keys=True).encode("utf-8")
).hexdigest()
source_07_run_spec_sha256 = hashlib.sha256(
    json.dumps(source_07_spec, sort_keys=True).encode("utf-8")
).hexdigest()

assert source_07_spec["model_key_sha256"] == model_key_hash
assert source_07_spec["feature_manifest_sha256"] == manifest_hash
assert source_07_audit["run_spec_sha256"] == source_07_run_spec_sha256
assert int(source_07_spec["outer_splits"]) == int(RIDGE_OUTER_SPLITS) == 5
assert int(source_07_spec["inner_splits"]) == int(RIDGE_INNER_SPLITS) == 3
assert [float(x) for x in source_07_spec["alpha_grid"]] == [float(x) for x in RIDGE_ALPHA_GRID]

print("Notebook-07 prerequisite gates and fingerprints: PASS")
print("Model-key SHA256:", model_key_hash)
print("Manifest SHA256:", manifest_hash)
print("Source Notebook-07 run-spec SHA256:", source_07_run_spec_sha256)
display(pd.Series(task_counts, name="n"))

## 2. Select a representative set of models

The subset is intentionally smaller than the full controls-plus-representation matrix because the purpose is to compare evaluation designs rather than repeat model selection. It includes five PTAL specifications, four compact-control EPC specifications and three richer-control EPC specifications. The selected feature columns must exactly match their definitions in the earlier analysis.

In [ ]:
# Reconstruct the 12 representative model specifications.
selected_definitions = [
    {
        "task": "PTAL",
        "base_feature_set": "PTAL_spatial_baseline",
        "added_feature_set": None,
        "analysis_role": "control_baseline",
    },
    {
        "task": "PTAL",
        "base_feature_set": "PTAL_spatial_baseline",
        "added_feature_set": "DINOv2",
        "analysis_role": "aerial_single",
    },
    {
        "task": "PTAL",
        "base_feature_set": "PTAL_spatial_baseline",
        "added_feature_set": "StreetView_CLIP_plus_metadata",
        "analysis_role": "streetview_content_plus_coverage",
    },
    {
        "task": "PTAL",
        "base_feature_set": "PTAL_spatial_baseline",
        "added_feature_set": "Street_and_sky",
        "analysis_role": "compact_fusion",
    },
    {
        "task": "PTAL",
        "base_feature_set": "PTAL_spatial_baseline",
        "added_feature_set": "All_representations_plus_SV_metadata",
        "analysis_role": "full_fusion",
    },
    {
        "task": "EPC",
        "base_feature_set": "EPC_controls_sparse",
        "added_feature_set": None,
        "analysis_role": "sparse_control_baseline",
    },
    {
        "task": "EPC",
        "base_feature_set": "EPC_controls_sparse",
        "added_feature_set": "DINOv2",
        "analysis_role": "aerial_single",
    },
    {
        "task": "EPC",
        "base_feature_set": "EPC_controls_sparse",
        "added_feature_set": "Sky_and_space",
        "analysis_role": "compact_fusion",
    },
    {
        "task": "EPC",
        "base_feature_set": "EPC_controls_sparse",
        "added_feature_set": "All_representations_plus_SV_metadata",
        "analysis_role": "full_fusion",
    },
    {
        "task": "EPC",
        "base_feature_set": "EPC_controls_extensive",
        "added_feature_set": None,
        "analysis_role": "privileged_control_baseline",
    },
    {
        "task": "EPC",
        "base_feature_set": "EPC_controls_extensive",
        "added_feature_set": "TESSERA",
        "analysis_role": "small_positive_representation_check",
    },
    {
        "task": "EPC",
        "base_feature_set": "EPC_controls_extensive",
        "added_feature_set": "All_representations_plus_SV_metadata",
        "analysis_role": "full_fusion_redundancy_check",
    },
]

def dedupe_preserve_order(columns):
    return list(dict.fromkeys(columns))

def columns_sha256(columns):
    return hashlib.sha256(
        json.dumps(list(columns), separators=(",", ":")).encode("utf-8")
    ).hexdigest()

source_model_specs = {
    row["model_id"]: row for row in source_07_spec["model_specifications"]
}
selected_records = []
selected_model_features = {}

for definition in selected_definitions:
    task = definition["task"]
    base = definition["base_feature_set"]
    added = definition["added_feature_set"]
    assert base in feature_sets
    if added is None:
        model_id = base
        columns = list(feature_sets[base])
    else:
        assert added in feature_sets
        model_id = f"{base}__plus__{added}"
        columns = dedupe_preserve_order(feature_sets[base] + feature_sets[added])

    assert model_id in source_model_specs, f"Model absent from Notebook 07: {model_id}"
    source_row = source_model_specs[model_id]
    assert source_row["task"] == task
    assert int(source_row["n_features_manifest"]) == len(columns)
    assert source_row["feature_columns_sha256"] == columns_sha256(columns)
    assert not [c for c in columns if c not in df.columns]

    selected_model_features[model_id] = columns
    selected_records.append({
        **definition,
        "model_id": model_id,
        "n_features_manifest": len(columns),
        "feature_columns_sha256": columns_sha256(columns),
    })

selected_specs = pd.DataFrame(selected_records)
assert len(selected_specs) == 12
assert selected_specs["model_id"].is_unique
assert selected_specs.groupby("task").size().to_dict() == {"EPC": 7, "PTAL": 5}

display(selected_specs)
print("Frozen sensitivity models:", len(selected_specs))

## 3. Retain the same Street View treatment

Street View visual values and coverage metadata are handled in the same way as in the main benchmark, ensuring that the only intended difference is the way train and test observations are divided.

In [ ]:
# Retain the established treatment of Street View non-coverage.
SV_META_COLS = [
    "sv_has_streetview", "sv_n_images", "sv_min_dist_m", "sv_mean_dist_m"
]
SV_CLIP_COLS = feature_sets["StreetView_CLIP_only"]

def apply_structural_sv_metadata_fill(task_df, task):
    out = task_df.copy()
    radius = {
        "PTAL": float(STREETVIEW_PTAL_RADIUS_M),
        "EPC": float(STREETVIEW_EPC_RADIUS_M),
    }[task]

    has = pd.to_numeric(out["sv_has_streetview"], errors="coerce")
    clip_complete = out[SV_CLIP_COLS].notna().all(axis=1)
    clip_all_missing = out[SV_CLIP_COLS].isna().all(axis=1)
    has = has.where(has.notna(), clip_complete.astype(int)).astype(int)

    assert has.isin([0, 1]).all()
    assert clip_complete[has.eq(1)].all()
    assert clip_all_missing[has.eq(0)].all()

    out["sv_has_streetview"] = has
    out["sv_n_images"] = pd.to_numeric(out["sv_n_images"], errors="coerce")
    out["sv_min_dist_m"] = pd.to_numeric(out["sv_min_dist_m"], errors="coerce")
    out["sv_mean_dist_m"] = pd.to_numeric(out["sv_mean_dist_m"], errors="coerce")

    no_sv = has.eq(0)
    out.loc[no_sv, "sv_n_images"] = 0.0
    for c in ["sv_min_dist_m", "sv_mean_dist_m"]:
        out.loc[no_sv & out[c].isna(), c] = radius

    assert out.loc[no_sv, "sv_n_images"].eq(0).all()
    assert out.loc[no_sv, ["sv_min_dist_m", "sv_mean_dist_m"]].notna().all().all()
    return out

sv_audit_rows = []
for task in ["PTAL", "EPC"]:
    task_filled = apply_structural_sv_metadata_fill(df[df["task"] == task], task)
    has = task_filled["sv_has_streetview"].eq(1)
    sv_audit_rows.append({
        "task": task,
        "n": int(len(task_filled)),
        "coverage_pct": float(100 * has.mean()),
        "n_no_streetview": int((~has).sum()),
    })
display(pd.DataFrame(sv_audit_rows))

## 4. Retain the same Ridge training pipeline

Numerical and categorical preparation remains inside each training split. Random division does not justify processing the complete dataset before testing; doing so would allow information from held-out observations to influence the model.

In [ ]:
# Build the same training-only preprocessing and Ridge pipeline.
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_pipeline(feature_cols):
    categorical_cols = [c for c in feature_cols if c in categorical_master]
    numeric_cols = [c for c in feature_cols if c not in categorical_master]
    transformers = []

    if numeric_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ))
    if categorical_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("onehot", make_onehot()),
            ]),
            categorical_cols,
        ))

    pre = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.0,
    )
    return Pipeline([
        ("preprocess", pre),
        ("ridge", Ridge(solver="lsqr", max_iter=5000, tol=1e-4)),
    ])

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def atomic_csv(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def atomic_parquet(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_parquet(tmp, index=False)
    tmp.replace(path)

def atomic_json(obj, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

def coerce_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    mapped = series.astype(str).str.strip().str.lower().map({"true": True, "false": False})
    assert mapped.notna().all()
    return mapped.astype(bool)

## 5. Create a shared random division

Within each task, observations are shuffled with a fixed seed and divided into five parts. All selected models use the same assignment. Boroughs are intentionally mixed between training and test sets because this is the evaluation condition being compared.

In [ ]:
# Create one fixed random five-part assignment shared by all selected models.
random_assignment_rows = []
random_fold_index_by_task = {}
random_fold_qa_rows = []

for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    splitter = KFold(
        n_splits=RIDGE_OUTER_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    task_fold = np.full(len(task_df), -1, dtype=int)

    for fold, (train_idx, test_idx) in enumerate(splitter.split(task_df)):
        task_fold[test_idx] = fold
        train_boroughs = set(task_df.iloc[train_idx][group_col].astype(str))
        test_boroughs = set(task_df.iloc[test_idx][group_col].astype(str))
        overlap = train_boroughs.intersection(test_boroughs)
        random_fold_qa_rows.append({
            "task": task,
            "random_outer_fold": int(fold),
            "n_train": int(len(train_idx)),
            "n_test": int(len(test_idx)),
            "n_train_boroughs": int(len(train_boroughs)),
            "n_test_boroughs": int(len(test_boroughs)),
            "n_boroughs_in_both": int(len(overlap)),
            "target_train_mean": float(task_df.iloc[train_idx][target_col].mean()),
            "target_test_mean": float(task_df.iloc[test_idx][target_col].mean()),
        })
        for idx in test_idx:
            random_assignment_rows.append({
                "sample_id": task_df.loc[idx, "sample_id"],
                "task": task,
                "random_outer_fold": int(fold),
                "borough_code": task_df.loc[idx, group_col],
                "target": float(task_df.loc[idx, target_col]),
            })

    assert (task_fold >= 0).all()
    random_fold_index_by_task[task] = task_fold

random_outer_folds = (
    pd.DataFrame(random_assignment_rows)
    .sort_values(["task", "sample_id"], kind="mergesort")
    .reset_index(drop=True)
)
assert len(random_outer_folds) == len(df)
assert random_outer_folds["sample_id"].is_unique

random_fold_qa = pd.DataFrame(random_fold_qa_rows)
assert random_fold_qa["n_boroughs_in_both"].gt(0).all()

if RANDOM_OUTER_FOLDS_PATH.exists():
    existing_random_folds = (
        pd.read_csv(RANDOM_OUTER_FOLDS_PATH)
        .sort_values(["task", "sample_id"], kind="mergesort")
        .reset_index(drop=True)
    )
    pd.testing.assert_frame_equal(
        existing_random_folds[random_outer_folds.columns],
        random_outer_folds,
        check_dtype=False,
        check_exact=False,
        rtol=0,
        atol=1e-12,
    )
    print("Verified existing frozen random-fold assignments.")
else:
    atomic_csv(random_outer_folds, RANDOM_OUTER_FOLDS_PATH)
    print("Saved frozen random-fold assignments:", RANDOM_OUTER_FOLDS_PATH)

random_fold_hash = hashlib.sha256(
    pd.util.hash_pandas_object(random_outer_folds, index=False).values.tobytes()
).hexdigest()

selected_model_payload = [
    {
        "task": row.task,
        "model_id": row.model_id,
        "base_feature_set": row.base_feature_set,
        "added_feature_set": (
            row.added_feature_set if pd.notna(row.added_feature_set) else None
        ),
        "analysis_role": row.analysis_role,
        "n_features_manifest": int(row.n_features_manifest),
        "feature_columns_sha256": row.feature_columns_sha256,
    }
    for row in selected_specs.itertuples(index=False)
]

run_spec = {
    "run_spec_version": "08-v1-2026-08-23",
    "notebook": "08_random_vs_spatial_cv_sensitivity.ipynb",
    "purpose": "diagnose random-split optimism relative to borough-grouped spatial CV",
    "source_notebook_07_run_spec": str(INCREMENTAL_RUN_SPEC_PATH),
    "source_notebook_07_audit": str(INCREMENTAL_AUDIT_PATH),
    "source_07_run_spec_sha256": source_07_run_spec_sha256,
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "random_outer_fold_assignment_sha256": random_fold_hash,
    "random_outer_splitter": {
        "class": "KFold",
        "n_splits": int(RIDGE_OUTER_SPLITS),
        "shuffle": True,
        "random_state": int(RANDOM_STATE),
    },
    "random_inner_splitter": {
        "class": "KFold",
        "n_splits": int(RIDGE_INNER_SPLITS),
        "shuffle": True,
        "random_state_rule": "RANDOM_STATE + 1000 + outer_fold",
    },
    "spatial_comparator": "frozen Notebook-07 borough-grouped nested CV",
    "alpha_grid": [float(x) for x in RIDGE_ALPHA_GRID],
    "inner_selection_metric": "RMSE",
    "selected_model_specifications": selected_model_payload,
    "comparison_metrics": [
        "random_minus_spatial_mean_r2",
        "spatial_minus_random_mean_rmse",
        "spatial_minus_random_mean_mae",
    ],
    "inference": "descriptive protocol comparison; random and spatial fold IDs are not paired",
    "dinov3_status": "out of remaining dissertation scope",
}
run_spec_sha256 = hashlib.sha256(
    json.dumps(run_spec, sort_keys=True).encode("utf-8")
).hexdigest()

if RANDOM_CV_RUN_SPEC_PATH.exists():
    with open(RANDOM_CV_RUN_SPEC_PATH, "r") as f:
        existing_spec = json.load(f)
    assert existing_spec == run_spec, (
        "Existing Notebook-08 checkpoints belong to a different run specification. "
        "Do not mix outputs; archive the old 08 outputs before a documented rerun."
    )
else:
    atomic_json(run_spec, RANDOM_CV_RUN_SPEC_PATH)

expected_runs_df = pd.DataFrame([
    {"task": row.task, "model_id": row.model_id, "random_outer_fold": fold}
    for row in selected_specs.itertuples(index=False)
    for fold in range(RIDGE_OUTER_SPLITS)
])
expected_keys = set(map(
    tuple,
    expected_runs_df[["task", "model_id", "random_outer_fold"]].to_numpy(),
))
expected_prediction_rows = int(sum(
    int(task_counts[task]) * int((selected_specs["task"] == task).sum())
    for task in ["PTAL", "EPC"]
))

assert len(expected_keys) == 60
assert expected_prediction_rows == 172985

display(random_fold_qa)
print("Random-fold SHA256:", random_fold_hash)
print("Notebook-08 run-spec SHA256:", run_spec_sha256)
print("Expected random-CV runs:", len(expected_keys))
print("Expected random-CV predictions:", expected_prediction_rows)

## 6. Fit the models under random evaluation

Each outer test part is predicted by a model trained on the other four. The Ridge penalty is chosen through further random divisions within the training data. Per-round results are saved so the analysis can continue after a disconnected session without mixing different specifications.

In [ ]:
# Fit each selected model under random evaluation and save round-level outputs.
if RANDOM_CV_RESULTS_PATH.exists():
    completed = pd.read_csv(RANDOM_CV_RESULTS_PATH)
    required_result_cols = {
        "task", "model_id", "random_outer_fold", "run_spec_sha256"
    }
    assert required_result_cols.issubset(completed.columns)
    assert not completed.duplicated(
        ["task", "model_id", "random_outer_fold"]
    ).any()
    assert completed["run_spec_sha256"].eq(run_spec_sha256).all()
    completed["random_outer_fold"] = completed["random_outer_fold"].astype(int)
    completed_keys = set(map(
        tuple,
        completed[["task", "model_id", "random_outer_fold"]].to_numpy(),
    ))
    assert completed_keys.issubset(expected_keys)
else:
    completed = pd.DataFrame()

result_rows = [] if completed.empty else completed.to_dict("records")

def checkpoint_is_valid(path, task, model_id, outer_fold, expected_ids):
    if not path.exists():
        return False
    try:
        p = pd.read_parquet(path)
        required = {
            "sample_id", "task", "model_id", "random_outer_fold",
            "borough_code", "y_true", "y_pred", "run_spec_sha256",
        }
        if not required.issubset(p.columns) or p["sample_id"].duplicated().any():
            return False
        if not p["task"].eq(task).all() or not p["model_id"].eq(model_id).all():
            return False
        if not p["random_outer_fold"].astype(int).eq(outer_fold).all():
            return False
        if not p["run_spec_sha256"].eq(run_spec_sha256).all():
            return False
        if not np.isfinite(p["y_true"]).all() or not np.isfinite(p["y_pred"]).all():
            return False
        return set(p["sample_id"].astype(str)) == set(pd.Series(expected_ids).astype(str))
    except Exception:
        return False

for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    task_df = apply_structural_sv_metadata_fill(task_df, task)
    y = pd.to_numeric(task_df[target_col], errors="raise").to_numpy()
    boroughs = task_df[group_col].astype(str).to_numpy()
    task_fold = random_fold_index_by_task[task]

    task_specs = selected_specs[selected_specs["task"] == task]
    for spec_row in task_specs.itertuples(index=False):
        model_id = spec_row.model_id
        cols = selected_model_features[model_id]
        X = task_df[cols]

        for outer_fold in range(RIDGE_OUTER_SPLITS):
            test_idx = np.flatnonzero(task_fold == outer_fold)
            train_idx = np.flatnonzero(task_fold != outer_fold)
            run_key = (task, model_id, outer_fold)
            pred_file = (
                RANDOM_CV_CHUNK_DIR
                / f"random__{task}__{model_id}__fold{outer_fold}.parquet"
            )

            completed_keys_now = {
                (r["task"], r["model_id"], int(r["random_outer_fold"]))
                for r in result_rows
            }
            checkpoint_ok = checkpoint_is_valid(
                pred_file, task, model_id, outer_fold,
                task_df.iloc[test_idx]["sample_id"].to_numpy(),
            )
            if run_key in completed_keys_now and checkpoint_ok:
                print("SKIP validated checkpoint:", run_key)
                continue

            print("\n" + "=" * 100)
            print("RANDOM CV |", task, "|", model_id, "| outer fold", outer_fold)
            print("=" * 100)

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            assert set(train_idx).isdisjoint(set(test_idx))

            inner = KFold(
                n_splits=RIDGE_INNER_SPLITS,
                shuffle=True,
                random_state=RANDOM_STATE + 1000 + outer_fold,
            )
            inner_splits = list(inner.split(X_train))
            for inner_train, inner_valid in inner_splits:
                assert set(inner_train).isdisjoint(set(inner_valid))

            pipe = build_pipeline(cols)
            search = GridSearchCV(
                estimator=pipe,
                param_grid={"ridge__alpha": RIDGE_ALPHA_GRID},
                scoring="neg_root_mean_squared_error",
                cv=inner_splits,
                refit=True,
                n_jobs=1,
                return_train_score=False,
                error_score="raise",
            )

            t0 = time.time()
            search.fit(X_train, y_train)
            elapsed_s = time.time() - t0
            pred = search.predict(X_test)
            assert len(pred) == len(test_idx)
            assert np.isfinite(pred).all()

            best_alpha = float(search.best_params_["ridge__alpha"])
            alpha_edge = best_alpha in {
                float(min(RIDGE_ALPHA_GRID)), float(max(RIDGE_ALPHA_GRID))
            }
            train_boroughs = set(boroughs[train_idx])
            test_boroughs = set(boroughs[test_idx])
            row = {
                "task": task,
                "model_id": model_id,
                "analysis_role": spec_row.analysis_role,
                "random_outer_fold": int(outer_fold),
                "validation_scheme": "random_nested_5fold",
                "n_features_manifest": int(len(cols)),
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
                "n_train_boroughs": int(len(train_boroughs)),
                "n_test_boroughs": int(len(test_boroughs)),
                "n_boroughs_in_both": int(len(train_boroughs.intersection(test_boroughs))),
                "best_alpha": best_alpha,
                "alpha_grid_edge": bool(alpha_edge),
                "inner_best_rmse": float(-search.best_score_),
                "r2": float(r2_score(y_test, pred)),
                "rmse": rmse(y_test, pred),
                "mae": float(mean_absolute_error(y_test, pred)),
                "fit_seconds": float(elapsed_s),
                "run_spec_sha256": run_spec_sha256,
            }

            pred_frame = pd.DataFrame({
                "sample_id": task_df.iloc[test_idx]["sample_id"].to_numpy(),
                "task": task,
                "model_id": model_id,
                "random_outer_fold": outer_fold,
                "borough_code": boroughs[test_idx],
                "y_true": y_test,
                "y_pred": pred,
                "run_spec_sha256": run_spec_sha256,
            })
            atomic_parquet(pred_frame, pred_file)

            result_rows = [
                r for r in result_rows
                if (
                    r["task"], r["model_id"], int(r["random_outer_fold"])
                ) != run_key
            ]
            result_rows.append(row)
            results_now = pd.DataFrame(result_rows).sort_values(
                ["task", "model_id", "random_outer_fold"], kind="mergesort"
            )
            atomic_csv(results_now, RANDOM_CV_RESULTS_PATH)
            print(row)

            del search, pipe, X_train, X_test, pred, pred_frame
            gc.collect()

print("Notebook-08 random-CV analysis is complete or safely checkpointed.")

## 7. Assemble random and borough predictions

All 60 random model rounds must be present before comparison. The corresponding borough results are then read for the same models and verified to cover the expected samples.

In [ ]:
# Verify complete random predictions and load the matching borough predictions.
random_results = pd.read_csv(RANDOM_CV_RESULTS_PATH)
random_results["random_outer_fold"] = random_results["random_outer_fold"].astype(int)
assert not random_results.duplicated(
    ["task", "model_id", "random_outer_fold"]
).any()
assert random_results["run_spec_sha256"].eq(run_spec_sha256).all()
actual_keys = set(map(
    tuple,
    random_results[["task", "model_id", "random_outer_fold"]].to_numpy(),
))

missing_keys = sorted(expected_keys - actual_keys)
unexpected_keys = sorted(actual_keys - expected_keys)
print("Completed random fold-runs:", len(actual_keys), "/", len(expected_keys))
print("Missing:", len(missing_keys), "| Unexpected:", len(unexpected_keys))
assert not missing_keys, "Notebook 08 incomplete: rerun Section 6."
assert not unexpected_keys

random_pred_frames = []
chunk_audit_rows = []
for task, model_id, outer_fold in sorted(expected_keys):
    path = (
        RANDOM_CV_CHUNK_DIR
        / f"random__{task}__{model_id}__fold{outer_fold}.parquet"
    )
    assert path.exists(), f"Missing random prediction chunk: {path.name}"
    p = pd.read_parquet(path)
    required = {
        "sample_id", "task", "model_id", "random_outer_fold",
        "borough_code", "y_true", "y_pred", "run_spec_sha256",
    }
    assert required.issubset(p.columns)
    assert p["task"].eq(task).all()
    assert p["model_id"].eq(model_id).all()
    assert p["random_outer_fold"].astype(int).eq(outer_fold).all()
    assert p["run_spec_sha256"].eq(run_spec_sha256).all()
    assert p["sample_id"].is_unique
    assert np.isfinite(p["y_true"]).all() and np.isfinite(p["y_pred"]).all()

    expected_fold = random_outer_folds[
        (random_outer_folds["task"] == task)
        & (random_outer_folds["random_outer_fold"] == outer_fold)
    ][["sample_id", "borough_code", "target"]].sort_values(
        "sample_id"
    ).reset_index(drop=True)
    observed_fold = p[
        ["sample_id", "borough_code", "y_true"]
    ].sort_values("sample_id").reset_index(drop=True)
    assert expected_fold["sample_id"].equals(observed_fold["sample_id"])
    assert expected_fold["borough_code"].astype(str).equals(
        observed_fold["borough_code"].astype(str)
    )
    assert np.allclose(
        expected_fold["target"], observed_fold["y_true"], rtol=0, atol=1e-12
    )

    chunk_audit_rows.append({
        "task": task,
        "model_id": model_id,
        "random_outer_fold": int(outer_fold),
        "n_rows": int(len(p)),
        "file": path.name,
    })
    random_pred_frames.append(p)

random_preds = pd.concat(random_pred_frames, ignore_index=True)
assert not random_preds.duplicated(["sample_id", "task", "model_id"]).any()
assert len(random_preds) == expected_prediction_rows

selected_ids = set(selected_specs["model_id"])
spatial_results_all = pd.read_csv(INCREMENTAL_RESULTS_PATH)
spatial_results = spatial_results_all[
    spatial_results_all["model_id"].isin(selected_ids)
].copy()
spatial_results["outer_fold"] = spatial_results["outer_fold"].astype(int)
assert len(spatial_results) == len(expected_keys)
assert spatial_results.groupby(["task", "model_id"]).size().eq(RIDGE_OUTER_SPLITS).all()
assert spatial_results["run_spec_sha256"].eq(source_07_run_spec_sha256).all()

spatial_preds_all = pd.read_parquet(INCREMENTAL_PREDICTIONS_PATH)
spatial_preds = spatial_preds_all[
    spatial_preds_all["model_id"].isin(selected_ids)
].copy()
assert len(spatial_preds) == expected_prediction_rows
assert not spatial_preds.duplicated(["sample_id", "task", "model_id"]).any()
assert spatial_preds["run_spec_sha256"].eq(source_07_run_spec_sha256).all()

for row in selected_specs.itertuples(index=False):
    n_expected = task_counts[row.task]
    assert len(random_preds[
        (random_preds["task"] == row.task)
        & (random_preds["model_id"] == row.model_id)
    ]) == n_expected
    assert len(spatial_preds[
        (spatial_preds["task"] == row.task)
        & (spatial_preds["model_id"] == row.model_id)
    ]) == n_expected

print("Validated random chunks:", len(chunk_audit_rows))
print("Validated random prediction rows:", len(random_preds))
print("Validated selected spatial result rows:", len(spatial_results))
print("Validated selected spatial prediction rows:", len(spatial_preds))

## 8. Measure the difference between evaluation designs

The reported R² difference is random minus borough performance, so a positive value means that random evaluation is more optimistic. Error differences are defined in the opposite order—borough error minus random error—so positive values have the same interpretation.

The five random parts and five borough parts contain different observations and geographical structures. Round numbers are therefore not paired, and no round-level significance test is applied.

In [ ]:
# Summarise each evaluation setting and calculate the optimism gap.
def summarise_scheme(fold_results, predictions, scheme, fold_col):
    rows = []
    for spec in selected_specs.itertuples(index=False):
        g = fold_results[
            (fold_results["task"] == spec.task)
            & (fold_results["model_id"] == spec.model_id)
        ]
        p = predictions[
            (predictions["task"] == spec.task)
            & (predictions["model_id"] == spec.model_id)
        ]
        assert len(g) == RIDGE_OUTER_SPLITS
        assert g[fold_col].nunique() == RIDGE_OUTER_SPLITS
        assert len(p) == task_counts[spec.task]

        rows.append({
            "task": spec.task,
            "model_id": spec.model_id,
            "analysis_role": spec.analysis_role,
            "base_feature_set": spec.base_feature_set,
            "added_feature_set": spec.added_feature_set,
            "n_features": int(spec.n_features_manifest),
            "validation_scheme": scheme,
            "mean_r2": float(g["r2"].mean()),
            "sd_r2": float(g["r2"].std(ddof=1)),
            "mean_rmse": float(g["rmse"].mean()),
            "sd_rmse": float(g["rmse"].std(ddof=1)),
            "mean_mae": float(g["mae"].mean()),
            "sd_mae": float(g["mae"].std(ddof=1)),
            "pooled_r2": float(r2_score(p["y_true"], p["y_pred"])),
            "pooled_rmse": rmse(p["y_true"], p["y_pred"]),
            "pooled_mae": float(mean_absolute_error(p["y_true"], p["y_pred"])),
            "alpha_edge_hits": int(coerce_bool(g["alpha_grid_edge"]).sum()),
            "median_best_alpha": float(g["best_alpha"].median()),
            "total_fit_minutes": float(g["fit_seconds"].sum() / 60),
        })
    return pd.DataFrame(rows)

random_summary = summarise_scheme(
    random_results, random_preds, "random_nested_5fold", "random_outer_fold"
)
spatial_summary = summarise_scheme(
    spatial_results, spatial_preds, "borough_grouped_nested_5fold", "outer_fold"
)

identity_cols = [
    "task", "model_id", "analysis_role", "base_feature_set",
    "added_feature_set", "n_features",
]
comparison = random_summary.merge(
    spatial_summary,
    on=identity_cols,
    how="inner",
    validate="one_to_one",
    suffixes=("_random", "_spatial"),
)
assert len(comparison) == len(selected_specs) == 12

comparison["random_minus_spatial_mean_r2"] = (
    comparison["mean_r2_random"] - comparison["mean_r2_spatial"]
)
comparison["random_minus_spatial_pooled_r2"] = (
    comparison["pooled_r2_random"] - comparison["pooled_r2_spatial"]
)
comparison["spatial_minus_random_mean_rmse"] = (
    comparison["mean_rmse_spatial"] - comparison["mean_rmse_random"]
)
comparison["spatial_minus_random_pooled_rmse"] = (
    comparison["pooled_rmse_spatial"] - comparison["pooled_rmse_random"]
)
comparison["spatial_minus_random_mean_mae"] = (
    comparison["mean_mae_spatial"] - comparison["mean_mae_random"]
)
comparison["spatial_minus_random_pooled_mae"] = (
    comparison["pooled_mae_spatial"] - comparison["pooled_mae_random"]
)

comparison = comparison.sort_values(
    ["task", "base_feature_set", "analysis_role"], kind="mergesort"
).reset_index(drop=True)

atomic_csv(comparison, RANDOM_VS_SPATIAL_SUMMARY_PATH)
atomic_parquet(random_preds, RANDOM_CV_PREDICTIONS_PATH)

display(comparison[[
    "task", "model_id", "analysis_role",
    "mean_r2_random", "mean_r2_spatial", "random_minus_spatial_mean_r2",
    "mean_rmse_random", "mean_rmse_spatial", "spatial_minus_random_mean_rmse",
    "mean_mae_random", "mean_mae_spatial", "spatial_minus_random_mean_mae",
]])
print("Saved:", RANDOM_VS_SPATIAL_SUMMARY_PATH)
print("Saved:", RANDOM_CV_PREDICTIONS_PATH)

## 9. Finalise the comparison

The completed output is checked for the full model-by-round matrix, exact prediction coverage and an adequate Ridge penalty range. The interpretation is determined by the observed direction and size of the protocol difference, not by requiring random evaluation to produce a particular result.

In [ ]:
# Validate the comparison and save the final protocol-sensitivity outputs.
counts = (
    random_results.groupby(["task", "model_id"])
    .size().rename("n_outer_folds").reset_index()
)
edge_counts = (
    random_results.assign(
        alpha_grid_edge=coerce_bool(random_results["alpha_grid_edge"])
    )
    .groupby(["task", "model_id"])["alpha_grid_edge"]
    .sum().rename("edge_hits").reset_index()
)
repeated_edge = edge_counts[edge_counts["edge_hits"] >= 3].copy()

audit = {
    "run_spec_path": str(RANDOM_CV_RUN_SPEC_PATH),
    "run_spec_sha256": run_spec_sha256,
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "random_outer_fold_assignment_sha256": random_fold_hash,
    "source_07_run_spec_sha256": source_07_run_spec_sha256,
    "source_07_integrity_gate_pass": bool(source_07_audit["integrity_gate_pass"]),
    "source_07_interpretation_gate_pass": bool(
        source_07_audit["interpretation_gate_pass"]
    ),
    "selected_models": int(len(selected_specs)),
    "expected_random_fold_runs": int(len(expected_keys)),
    "completed_random_fold_runs": int(len(actual_keys)),
    "all_models_have_expected_random_folds": bool(
        counts["n_outer_folds"].eq(RIDGE_OUTER_SPLITS).all()
        and len(counts) == len(selected_specs)
    ),
    "expected_random_prediction_rows": int(expected_prediction_rows),
    "actual_random_prediction_rows": int(len(random_preds)),
    "random_prediction_rows_match_expected": bool(
        len(random_preds) == expected_prediction_rows
    ),
    "random_prediction_chunks_validated": int(len(chunk_audit_rows)),
    "selected_spatial_fold_rows_validated": int(len(spatial_results)),
    "selected_spatial_prediction_rows_validated": int(len(spatial_preds)),
    "comparison_rows": int(len(comparison)),
    "random_folds_mix_boroughs_by_design": bool(
        random_fold_qa["n_boroughs_in_both"].gt(0).all()
    ),
    "n_models_random_mean_r2_above_spatial": int(
        comparison["random_minus_spatial_mean_r2"].gt(0).sum()
    ),
    "n_models_random_mean_r2_equal_or_below_spatial": int(
        comparison["random_minus_spatial_mean_r2"].le(0).sum()
    ),
    "n_alpha_grid_edge_hits": int(edge_counts["edge_hits"].sum()),
    "n_models_with_repeated_edge_hits": int(len(repeated_edge)),
    "alpha_grid_adequacy_pass": bool(repeated_edge.empty),
    "primary_spatial_result_source": "frozen Notebook 07",
    "comparison_inference": (
        "descriptive random-vs-borough protocol gap; fold IDs are not paired"
    ),
    "dinov3_status": "out of remaining dissertation scope",
    "integrity_gate_pass": True,
    "interpretation_gate_pass": bool(repeated_edge.empty),
}
atomic_json(audit, RANDOM_CV_AUDIT_PATH)

display(pd.Series(audit, name="value"))
if not repeated_edge.empty:
    display(repeated_edge)
    raise RuntimeError(
        "Integrity PASS, but interpretation is on hold: at least one random-CV "
        "model selected an alpha-grid boundary in >=3 outer folds. Expand only "
        "the alpha grid in a documented rerun."
    )

print("08 random-vs-spatial sensitivity — integrity gate: PASS")
print("08 random-vs-spatial sensitivity — interpretation gate: PASS")

## Interpretation

Randomly mixed train and test locations provide a modest but consistent uplift in apparent performance. This is compatible with local spatial similarity making interpolation easier than transfer to unseen borough groups. Because the uplift affects every selected model but leaves their ranking unchanged, the principal substantive conclusions are stable, while the borough-based scores offer the more cautious estimate of geographical generalisation.